# CLV — UCI Online Retail II EDA

**Prerequisite:** `python scripts/uci_pipeline.py`

Noncontractual retail spine (~5.8k customers, ~800k EU invoice lines).

| Parquet | Grain | Use |
|---------|-------|-----|
| `uci_fact_transactions` | Invoice line | Source for **order** spine |
| `uci_dim_customers` | Customer | Lifetime rollups / sanity checks |

**Next:** `00-customer-base-audit.ipynb` → `01-clv.ipynb` (BG/NBD + Gamma-Gamma on orders).


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "uci_pipeline.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
PALETTE = {"primary": "#1f4e79", "accent": "#c0392b", "neutral": "#7f8c8d"}
print("Config loaded ·", DATA)


In [ ]:
def load(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — run: python scripts/uci_pipeline.py")
    return pd.read_parquet(path)

fact = load("uci_fact_transactions.parquet")
fact["order_date"] = pd.to_datetime(fact["order_date"])
dim = load("uci_dim_customers.parquet")
orders = (
    fact.groupby(["order_id", "customer_id", "country"], as_index=False)
    .agg(order_revenue=("line_total", "sum"), order_lines=("line_total", "size"), order_date=("order_date", "min"))
)
print(f"Lines: {len(fact):,} · Orders: {orders['order_id'].nunique():,} · Customers: {fact['customer_id'].nunique():,}")
print(f"Span: {fact['order_date'].min().date()} → {fact['order_date'].max().date()}")
print(f"Total revenue: £{fact['line_total'].sum():,.0f}")
fact.head(3)


## 1 · Concentration (orders)


In [ ]:
cust = orders.groupby("customer_id")["order_revenue"].sum().sort_values(ascending=False)
cum = cust.cumsum() / cust.sum()
pct_cust = np.arange(1, len(cust) + 1) / len(cust)
half = float(pct_cust[np.searchsorted(cum.values, 0.5)])
print(f"Customers for 50% of revenue: {half:.1%}")
print(f"One-order customers (lifetime): {(dim['total_orders'] == 1).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(cust.clip(upper=cust.quantile(0.99)), bins=40, color=PALETTE["primary"], edgecolor="white")
axes[0].set_title("Customer lifetime spend")
axes[0].set_xlabel("£")
axes[1].plot(pct_cust * 100, cum * 100, color=PALETTE["accent"])
axes[1].axhline(50, ls=":", color="gray")
axes[1].set_title("Whale / Lorenz curve")
axes[1].set_xlabel("% customers (richest first)")
axes[1].set_ylabel("% revenue")
plt.tight_layout()
plt.show()


## 2 · Next steps


| Notebook | Role |
|----------|------|
| `00-customer-base-audit.ipynb` | Five Lenses customer×time audit |
| `01-clv.ipynb` | BG/NBD + Gamma-Gamma expected CLV |

Do not model CLV at invoice-line grain.
